# MMAStats presentation of the database

This notebook presents the structure and content of the **MMA Global Database**, a DuckDB dataset of 129,285 MMA fights covering the period 1993 to 2026, across more than 4,300 organizations.

The database is organized into six tables:

| Table | Rows | Purpose |
|---|---|---|
| `fighters_master` | 15,801 | Fighter dimension (physical attributes, nationality, gym) |
| `fights_career_longitudinal` | 129,285 | Main fact table covering all organizations |
| `fights_master_typed` | 28,118 | UFC subset with granular technical statistics |
| `records_career` | 346 | UFC official career records by fighter |
| `records_fight` | 572 | UFC official single-fight records |
| `records_event` | 142 | UFC official event-level records |

Each section below presents one table with two queries: the first describes its structure and basic content, the second extracts a relevant insight from the data.

All queries are written in SQL and executed via DuckDB inside this notebook.

In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/leandroiber/mmastats/dataset_global_v3.duckdb


In [2]:
# Install DuckDB if not present in the Kaggle environment
!pip install duckdb --quiet

import duckdb
import pandas as pd

# Path to the dataset file as exposed by the Kaggle runtime
DB_PATH = "/kaggle/input/datasets/leandroiber/mmastats/dataset_global_v3.duckdb"

# Open the database in read-only mode
con = duckdb.connect(DB_PATH, read_only=True)

# Helper function to execute a SQL query and return a DataFrame
def q(sql):
    return con.execute(sql).fetchdf()

# List the tables available in the database
q("SHOW TABLES")

,name
0,fighters_master
1,fights_career_longitudinal
2,fights_master_typed
3,records_career
4,records_event
5,records_fight
6,v_records_career_full
7,v_records_event_full
8,v_records_fight_full


## 1. fighters_master

This table is the fighter dimension. It contains one row per unique fighter, regardless of the organizations they competed in. Each fighter has a unique identifier (`fighter_id`) that is referenced across all the other tables.

Key fields include physical attributes (`height_cm`, `reach_cm`, `stance`), demographic information (`dob`, `nationality`), and team affiliation (`gym`).

Note on coverage: `reach_cm` is populated mostly for fighters who competed in the UFC, since the UFC is the only major organization that publishes official reach measurements. Roughly 70 percent of values are NULL.

In [3]:
# Sample of fighter records with main attributes
q("""
SELECT 
    fighter_name,
    nationality,
    gym,
    height_cm,
    reach_cm,
    stance
FROM fighters_master
WHERE reach_cm IS NOT NULL
ORDER BY reach_cm DESC
LIMIT 10
""")

,fighter_name,nationality,gym,height_cm,reach_cm,stance
0,Robelis Despaigne,None,None,200.66,213.36,Orthodox
1,Stefan Struve,None,None,210.82,213.36,Orthodox
2,Sergei Pavlovich,None,None,190.50,213.36,Southpaw
3,Jon Jones,None,None,193.04,213.36,Orthodox
4,Tallison Teixeira,Brazil,Team Lucas Mineiro,203.20,210.82,Orthodox
5,Oumar Sy,France,Bulgarian Top Team,193.04,210.82,Orthodox
6,Francis Ngannou,France,Xtreme Couture,193.04,210.82,Orthodox
7,Kennedy Nzechukwu,United States,Fortis MMA,195.58,210.82,Southpaw
8,Cheick Kongo,France,Kongo Smashin' Club,193.04,208.28,Orthodox
9,Lavar Johnson,United States,Pro Buhawe,193.04,208.28,Orthodox


In [4]:
# Top 10 nationalities by number of fighters in the database
q("""
SELECT 
    nationality,
    COUNT(*) AS fighters,
    ROUND(AVG(height_cm), 1) AS avg_height_cm
FROM fighters_master
WHERE nationality IS NOT NULL
GROUP BY nationality
ORDER BY fighters DESC
LIMIT 10
""")

,nationality,fighters,avg_height_cm
0,United States,3965,177.4
1,Brazil,2219,174.8
2,Russia,1668,176.5
3,England,912,177.8
4,Italy,472,175.5
5,Poland,436,179.5
6,Japan,342,171.1
7,Netherlands,230,179.8
8,France,226,177.1
9,Germany,211,177.8


## 2. fights_career_longitudinal

This is the main fact table of the database. It records every fight from every organization scraped by the project, totaling 129,285 fights from 4,353 distinct organizations.

Each row represents one fight and contains identification, both fighters, outcome (winner, method, round, finish time), weight class, and contextual fields such as referee, gym affiliations and nationalities at the time of the fight.

The boolean column `is_major_org` flags fights belonging to one of the 10 main organizations: UFC, Bellator, ACA, Cage Warriors, LFA, Jungle Fight, PFL, KSW, Oktagon and Rizin.

## 2. fights_career_longitudinal

This is the main fact table of the database. It records every fight from every organization scraped by the project, totaling 129,285 fights from 4,353 distinct organizations.

Each row represents one fight and contains identification, both fighters, outcome (winner, method, round, finish time), weight class, and contextual fields such as referee, gym affiliations and nationalities at the time of the fight.

The boolean column `is_major_org` flags fights belonging to one of the 10 main organizations: UFC, Bellator, ACA, Cage Warriors, LFA, Jungle Fight, PFL, KSW, Oktagon and Rizin.

In [5]:
# Volume of fights and events per major organization
q("""
SELECT 
    organization,
    COUNT(*) AS fights,
    COUNT(DISTINCT event_name) AS events,
    MIN(event_date) AS first_event,
    MAX(event_date) AS last_event
FROM fights_career_longitudinal
WHERE is_major_org
GROUP BY organization
ORDER BY fights DESC
""")

,organization,fights,events,first_event,last_event
0,aca,6040,279,2012-10-20,2026-06-18
1,bellator,5036,316,2009-04-03,2024-09-14
2,ufc,4563,797,1993-11-12,2026-05-28
3,cagewarriors,4027,329,2002-02-23,2026-07-04
4,lfa,3441,237,2017-01-13,2026-06-19
5,jungle_fight,2377,157,2003-09-12,2026-06-06
6,shooto,2344,721,1996-05-07,2026-05-24
7,ksw,1783,126,2004-02-27,2026-07-18
8,oktagon,1624,115,2012-03-24,2026-09-26
9,rizin,1347,82,2015-12-29,2026-09-09


## 3. fights_master_typed

This table is a detailed subset of UFC fights enriched with granular technical statistics from UFCStats.com. It contains 28,118 fights, of which 8,708 are UFC fights with full per-fighter statistics: significant strikes (landed and attempted), takedowns, knockdowns and ground control time.

The same fight present here can be cross-referenced with `fights_career_longitudinal` through the shared `fight_id`.

The column `is_title_fight` was externally validated against the Wikipedia article "List of UFC champions", reaching 99.2 percent match. The original flag is preserved as `is_title_fight_original` for auditability.

In [6]:
# Coverage of granular statistics for UFC fights only
q("""
SELECT 
    COUNT(*) AS total_ufc_fights,
    ROUND(100.0 * COUNT(*) FILTER (WHERE f1_sig_str_landed IS NOT NULL) / COUNT(*), 1) AS pct_sig_strikes,
    ROUND(100.0 * COUNT(*) FILTER (WHERE f1_td_landed IS NOT NULL) / COUNT(*), 1) AS pct_takedowns,
    ROUND(100.0 * COUNT(*) FILTER (WHERE f1_kd IS NOT NULL) / COUNT(*), 1) AS pct_knockdowns,
    ROUND(100.0 * COUNT(*) FILTER (WHERE f1_ctrl_seconds IS NOT NULL) / COUNT(*), 1) AS pct_control_time,
    ROUND(100.0 * COUNT(*) FILTER (WHERE f1_reach_cm IS NOT NULL) / COUNT(*), 1) AS pct_reach
FROM fights_master_typed
WHERE organization = 'ufc'
""")

,total_ufc_fights,pct_sig_strikes,pct_takedowns,pct_knockdowns,pct_control_time,pct_reach
0,8708,99.8,99.8,99.8,97.7,96.2


In [7]:
# Fights ranked by total significant strikes landed by both fighters combined
q("""
SELECT 
    fighter_1 || ' vs ' || fighter_2 AS fight,
    event_name,
    event_date,
    f1_sig_str_landed + f2_sig_str_landed AS total_sig_strikes,
    method_normalized AS result
FROM fights_master_typed
WHERE organization = 'ufc'
  AND f1_sig_str_landed IS NOT NULL
ORDER BY total_sig_strikes DESC
LIMIT 10
""")

,fight,event_name,event_date,total_sig_strikes,result
0,Max Holloway vs Calvin Kattar,UFC Fight Night: Holloway vs. Kattar,2021-01-16,578,Decision
1,Marlon Vera vs Rob Font,UFC Fight Night: Font vs. Vera,2022-04-30,430,Decision
2,Joshua Van vs Brandon Royval,UFC 317: Topuria vs. Oliveira,2025-06-28,419,Decision
3,Max Holloway vs Brian Ortega,UFC 231: Holloway vs. Ortega,2018-12-08,400,TKO
4,Jared Cannonier vs Marvin Vettori,UFC Fight Night: Vettori vs. Cannonier,2023-06-17,394,Decision
5,Max Holloway vs Yair Rodriguez,UFC Fight Night: Holloway vs. Rodriguez,2021-11-13,389,Decision
6,Dustin Poirier vs Max Holloway,UFC 236: Holloway vs. Poirier 2,2019-04-13,359,Decision
7,Shane Burgos vs Billy Quarantillo,UFC 268: Usman vs. Covington 2,2021-11-06,357,Decision
8,Marvin Vettori vs Paulo Costa,UFC Fight Night: Costa vs. Vettori,2021-10-23,353,Decision
9,Josh Hokit vs Curtis Blaydes,UFC 327: Prochazka vs. Ulberg,2026-04-11,351,Decision


## 4. records_career

This table holds UFC official career records scraped from `statleaders.ufc.com`. It contains 346 entries across 32 categories such as total fights, wins by knockout, wins by submission, title fight wins, and longest control time.

Each row has a `fighter_id` foreign key that links back to `fighters_master`, enabling joins to fighter biography and physical attributes. The column `valor` holds the original string value (which can be a number, a percentage or a time), while `valor_num` and `valor_segundos` provide typed numeric and time-in-seconds representations.

Coverage for this table is restricted to UFC because the UFC is the only major MMA organization that publishes consolidated career leaderboards.

In [8]:
# List of all 32 record categories in the career table
q("""
SELECT 
    categoria AS category,
    COUNT(*) AS entries
FROM records_career
GROUP BY categoria
ORDER BY entries DESC
LIMIT 15
""")

,category,entries
0,Vitórias por Finalização,19
1,Total de Lutas,14
2,Vitórias por Nocaute/ Nocaute Técnico,13
3,Tentativas de Finalização,13
4,Sequência de Vitórias,12
5,Nocautes/Finalizações,12
6,Knockdowns,11
7,Vitórias em Disputa de Cinturão,11
8,Taxa de precisão de Quedas,11
9,Vitórias,10


In [9]:
# Direct read from the official UFC submissions leaderboard
q("""
SELECT 
    rank,
    fighter_name,
    valor_num AS submissions
FROM records_career
WHERE categoria = 'Vitórias por Finalização'
ORDER BY rank
LIMIT 10
""")

,rank,fighter_name,submissions
0,1,Charles Oliveira,17.0
1,2,Jim Miller,14.0
2,3,Demian Maia,11.0
3,3,Gerald Meerschaert,11.0
4,5,Nate Diaz,10.0
5,6,Michael Chiesa,9.0
6,7,Frank Mir,8.0
7,7,Gunnar Nelson,8.0
8,7,Islam Makhachev,8.0
9,10,Aleksei Oleinik,7.0


## 5. records_fight

This table contains UFC official records measured at the level of a single fight (or a single round within a fight). It has 572 entries across 50 categories distributed in four scopes:

- `fight`: per-fighter metrics over the entire fight
- `fight_comb`: metrics combined between both fighters over the entire fight
- `round`: per-fighter metrics within a single round
- `round_comb`: metrics combined between both fighters within a single round

Each row stores both fighters (`fighter_a`, `fighter_b`), the event metadata, the value and a `fight_id` that links back to `fights_master_typed`.

Examples of categories include shortest fight, fastest knockout, fastest submission, most significant strikes landed in a fight, most takedowns in a fight, and so on.

In [10]:
# Fights ranked by shortest finish time
q("""
SELECT 
    rank,
    fighter_a || ' vs ' || fighter_b AS fight,
    valor AS time_to_finish,
    event_name,
    event_date
FROM records_fight
WHERE categoria = 'Luta mais curta'
ORDER BY rank
LIMIT 10
""")

,rank,fight,time_to_finish,event_name,event_date
0,1,Jorge Masvidal vs Ben Askren,0:05,UFC 239: Jones vs. Santos,2019-07-06
1,2,Duane Ludwig vs Jonathan Goulet,0:06,UFC Fight Night 3,2006-01-16
2,3,Chan Sung Jung vs Mark Hominick,0:07,UFC 140: Jones vs Machida,2011-12-10
3,3,Ryan Jimmo vs Anthony Perosh,0:07,UFC 149: Faber vs Barao,2012-07-21
4,3,Terrance McKinney vs Matt Frevola,0:07,UFC 263: Adesanya vs. Vettori 2,2021-06-12
5,3,Todd Duffee vs Tim Hague,0:07,UFC 102: Couture vs Nogueira,2009-08-29
6,7,James Irvin vs Houston Alexander,0:08,UFC Fight Night: Florian vs Lauzon,2008-04-02
7,7,Leon Edwards vs Seth Baczynski,0:08,UFC Fight Night: Gonzaga vs Cro Cop 2,2015-04-11
8,7,Makwan Amirkhani vs Andy Ogle,0:08,UFC on FOX: Gustafsson vs Johnson,2015-01-24
9,10,Gray Maynard vs Joe Veres,0:09,UFC Fight Night: Thomas vs Florian,2007-09-19


In [11]:
# Top fights by takedowns landed by a single fighter
q("""
SELECT 
    rank,
    fighter_a || ' vs ' || fighter_b AS fight,
    valor_num AS takedowns,
    event_name,
    event_date
FROM records_fight
WHERE categoria = 'Quedas Aplicadas'
  AND pagina = 'fight'
ORDER BY rank
LIMIT 10
""")

,rank,fight,takedowns,event_name,event_date
0,1,Khabib Nurmagomedov vs Abel Trujillo,21.0,UFC 160: Velasquez vs Silva 2,2013-05-25
1,2,Merab Dvalishvili vs Cory Sandhagen,20.0,UFC 320: Ankalaev vs. Pereira 2,2025-10-04
2,3,Myktybek Orolbai vs Chris Curtis,19.0,UFC Fight Night: Emmett vs. Vallejos,2026-03-14
3,4,Raul Rosas Jr. vs Rob Font,16.0,UFC 326: Holloway vs. Oliveira 2,2026-03-07
4,4,Sean Sherk vs Hermes Franca,16.0,UFC 73: Stacked,2007-07-07
5,6,Curtis Blaydes vs Alexander Volkov,14.0,UFC Fight Night: Blaydes vs. Volkov,2020-06-20
6,6,Demetrious Johnson vs Kyoji Horiguchi,14.0,UFC 186: Johnson vs Horiguchi,2015-04-25
7,8,Luigi Fioravanti vs Luke Cummo,13.0,UFC 82: Pride of a Champion,2008-03-01
8,8,Makhmud Muradov vs Bryan Barberena,13.0,UFC Fight Night: Aspinall vs. Tybura,2023-07-22
9,8,Merab Dvalishvili vs Gustavo Lopez,13.0,UFC Fight Night: Eye vs. Calvillo,2020-06-13


## 6. records_event

This table aggregates UFC official records at the event level: 142 entries across 11 categories. Examples include shortest event by total fight time, longest event, event with the most knockouts, event with the most significant strikes, and event with the most takedowns.

Each row stores the event name, event date, number of fights on the card, and the value of the metric. The column `event_matched` is TRUE when the event was successfully cross-referenced with `fights_master_typed`, which is the case for 100 percent of entries in this table.

In [12]:
# Events ranked by total knockouts (KO and TKO combined)
q("""
SELECT 
    rank,
    event_name,
    event_date,
    n_fights AS fights_on_card,
    valor_num AS knockouts
FROM records_event
WHERE categoria = 'Nocaute/ Nocaute Técnico'
ORDER BY rank
LIMIT 10
""")

,rank,event_name,event_date,fights_on_card,knockouts


In [13]:
# Events ranked by total cumulative fight time
q("""
SELECT 
    rank,
    event_name,
    event_date,
    n_fights AS fights_on_card,
    valor AS total_event_time
FROM records_event
WHERE categoria = 'Evento Mais Longo'
ORDER BY rank
LIMIT 10
""")

,rank,event_name,event_date,fights_on_card,total_event_time
0,1,UFC 263: Adesanya vs. Vettori 2,2021-06-12,14,3:19:32
1,2,UFC 251: Usman vs. Masvidal,2020-07-11,13,3:07:27
2,3,UFC Fight Night: Moreno vs. Albazi,2024-11-02,13,3:06:38
3,4,UFC Fight Night: Sterling vs. Zalal,2026-04-25,13,3:04:24
4,5,UFC Fight Night: Werdum vs. Tybura,2017-11-18,13,3:04:18
5,6,UFC 286: Edwards vs. Usman 3,2023-03-18,15,2:57:58
6,7,UFC Fight Night: Joanna vs. Waterson,2019-10-12,14,2:57:27
7,8,UFC 271: Adesanya vs. Whittaker 2,2022-02-12,14,2:57:22
8,9,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03,13,2:55:23
9,10,UFC Fight Night: Luque vs. Muhammad,2022-04-16,14,2:54:44


## Wrap-up

The six tables above form a complete relational view of the MMA Global Database. The two main fact tables, `fights_career_longitudinal` and `fights_master_typed`, are linked to the fighter dimension through `fighter_id`, and the three records tables provide an official benchmark layer for cross-validation against UFC public leaderboards.

For full documentation, methodology notes and the reproducible ELT pipeline, see the GitHub repository linked in the dataset description.

Suggested next steps for analysis:

- Temporal analysis of finish rates per decade
- Cross-organization fighter career trajectories
- Validation queries comparing the database against the official UFC records
- Distribution of physical attributes (height, reach, stance) by weight class